In [ ]:
import pandas as pa
import numpy as np
import matplotlib.pyplot as plt

# Chargement
# data = pa.read_csv('data.csv', delimiter=';', decimal='.') 

# Vérification des types et valeurs manquantes
# data.info()
# data.describe()

# Séparation Features (X) / Target (y)
X = data[['feature1', 'feature2', 'feature3']] # ou data.drop(['target'], axis=1)
y = data['target']

# Binarisation (si demandé, ex: transformer classe 2 en 1)
# y[y == 2] = 1

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# Note: fit sur le train, transform sur le train ET le test

In [ ]:
from sklearn.model_selection import train_test_split

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [ ]:
2. Classification : k-Nearest Neighbors (k-NN)

# 1. Distance Euclidienne
def euclidean(u, v):
    # Attention: v peut contenir la classe en dernière position
    return np.sqrt(np.sum((u - v[:-1])**2))

# 2. Calculer toutes les distances
def distances(u, dataset):
    dist = []
    for d in dataset:
        dist.append(euclidean(u, d))
    return dist

# 3. Trouver les k voisins
def voisins(u, dataset, k):
    dists = distances(u, dataset)
    indices_tries = np.argsort(dists) # trier les indices par distance croissante
    return dataset[indices_tries[:k]] # retourner les k premiers points

# 4. Prédire la classe (Vote majoritaire)
def classifier(u, dataset, k):
    k_voisins = voisins(u, dataset, k)
    classes = {} # Dictionnaire pour compter
    for v in k_voisins:
        classe = v[-1] # La classe est la dernière colonne
        classes[classe] = classes.get(classe, 0) + 1
    
    # Retourner la classe avec le max de votes
    return max(classes, key=classes.get)

#Version SKLearn
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=3, metric='euclidean')
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

#Best K
def best_k_search(X_train, X_test, y_train, y_test, max_k=10):
    best_acc = 0
    best_k = 1
    
    for k in range(1, max_k + 1):
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_train, y_train)
        score = knn.score(X_test, y_test)
        print(f'k={k}, accuracy={score:.2%}')
        
        if score > best_acc:
            best_acc = score
            best_k = k
            
    return best_k, best_acc

In [ ]:
3. Classification : Naive Bayes

import scipy.stats

# 1. Apprentissage : Calcul des moyennes et écarts-types par classe
X_c0 = X_train[y_train == 0]
X_c1 = X_train[y_train == 1]

mean_0, std_0 = np.mean(X_c0, axis=0), np.std(X_c0, axis=0)
mean_1, std_1 = np.mean(X_c1, axis=0), np.std(X_c1, axis=0)

# Probabilités a priori (P(C0) et P(C1))
p_c0 = len(X_c0) / len(X_train)
p_c1 = len(X_c1) / len(X_train)

# 2. Classification (MAP)
def map_classify(x):
    # Vraisemblance (Likelihood) * Prior
    # Pour dimension > 1, on multiplie les probas de chaque feature (hyp. naives)
    
    # Produit des PDF pour chaque feature pour la classe 0
    likelihood_0 = np.prod(scipy.stats.norm.pdf(x, mean_0, std_0))
    prob_0 = likelihood_0 * p_c0
    
    # Produit des PDF pour chaque feature pour la classe 1
    likelihood_1 = np.prod(scipy.stats.norm.pdf(x, mean_1, std_1))
    prob_1 = likelihood_1 * p_c1
    
    return 0 if prob_0 > prob_1 else 1

#version sklearn
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred = nb.predict(X_test)

In [ ]:
4. Métriques d'Évaluation (TD3)

def compute_confusion_matrix(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return np.array([[tp, fn], [fp, tn]]) # Attention à l'ordre conventionnel

# Accuracy: Taux de réussite global
accuracy = (tp + tn) / (tp + tn + fp + fn)

# Precision: Parmi ceux prédits positifs, combien le sont vraiment ?
precision = tp / (tp + fp)

# Recall (Rappel): Parmi ceux qui sont vraiment positifs, combien a-t-on trouvés ?
recall = tp / (tp + fn)

# F1-Score: Moyenne harmonique
f1 = 2 * (precision * recall) / (precision + recall)

#version sklearn
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Affichage graphique
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot()
plt.show()


In [ ]:
5. Réduction de Dimension ACP

def calculACP(X, d=2):
    n = X.shape[0]
    # 1. Matrice de Covariance (ou Scatter Matrix)
    # Données doivent être centrées réduites au préalable !
    cov_mat = np.cov(X.T) 
    
    # 2. Valeurs propres et Vecteurs propres
    eig_vals, eig_vecs = np.linalg.eig(cov_mat)
    
    # 3. Trier par valeur propre décroissante
    sorted_indices = np.argsort(eig_vals)[::-1]
    sorted_vecs = eig_vecs[:, sorted_indices]
    
    # 4. Garder les d premiers vecteurs
    W = sorted_vecs[:, :d]
    return W

def projection(W, X):
    return np.dot(X, W)

#version sklearn
from sklearn.decomposition import PCA

pca = PCA() # Pas de n_components pour tout voir
pca.fit(X_train_scaled) # Toujours sur données scalées

# Critère du coude (Scree plot)
plt.plot(np.arange(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_)
plt.ylabel('Variance expliquée')
plt.xlabel('Composantes')
plt.show()

# Critère variance cumulée (ex: garder 90%)
cumsum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumsum >= 0.90) + 1
print(f"Dimensions pour 90% variance : {d}")

# Application
pca = PCA(n_components=d)
X_train_proj = pca.fit_transform(X_train_scaled)
X_test_proj = pca.transform(X_test_scaled) # Transform seulement sur le test

REDUCTION LDA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis() # n_components = n_classes - 1 par défaut
X_train_lda = lda.fit_transform(X_train, y_train) # A besoin de y_train !
X_test_lda = lda.transform(X_test)

In [ ]:
6. Régression (TD2 & TD1)

import statsmodels.api as sm

# Ajouter la constante (Biais/Intercept) manuellement pour statsmodels
X_with_const = sm.add_constant(X)

model = sm.OLS(y, X_with_const)
results = model.fit()
print(results.summary())

# Interprétation :
# Si P>|t| (p-value) < 0.05 : La variable est significative.
# R-squared : % de variance expliquée par le modèle.

def gradient_descent(df, x0, y, epsilon=0.01, nu=0.001):
    # df : fonction dérivée
    # nu : learning rate (taux d'apprentissage)
    
    x_current = x0
    x_prec = x0 + 2 * epsilon # Juste pour entrer dans la boucle
    
    while abs(x_prec - x_current) > epsilon:
        x_prec = x_current
        derivee = df(x_prec)
        x_current = x_prec - nu * derivee
        
    return x_current